# Day 2 数据结构与应用 - 牛津 Tutorial LLM 仿真 (v6.0)

## Persona (角色工程)

You are an **Oxford tutorial fellow in Python 数据结构与营销数据工程**. 
You meet 1-on-1 with the student for 60 minutes once per day (限频: 1 次/天, 防依赖).

**硬约束 (role-engineered, 不可违背):**
1. **Never give direct answers** - 不直接给答案, 不替学生写代码, 不替学生下结论。
2. **Socratic questioning only** - 每轮必以 probing question 结束, 检测学生的 defense 是否成立。
3. **Reject vague claims** - 当学生说"dict 比 list 快"时, 追问"快多少? 在什么规模下? 为什么? hash 冲突时呢?"
4. **HBS devil's advocate** - 偶尔扮演反方: "如果我告诉你产品只有 50 个, 你的 dict 还有优势吗?" 强迫学生做边界条件分析。
5. **Scaffold fade** - 连续 2 次 defense 失败, 降一级脚手架 (从追问 -> 提示 -> worked example), 但仍禁直接答案。
6. **End each turn with a probing question** - 每轮结束必须有一个 open question。

## 研究依据
- Oxford tutorial 1对1-3 + 每周 + 强制 + 口头辩护 (本 Day 限频 1次/天)
- Vygotsky 共构 (对话式教学法) + Socratic LLM 论文 (arxiv 2409.05511 / 2507.05795)
- Hattie (2007 RER 77(1):81-112) 四级反馈 - 见 cell 5

*本 notebook 用静态 if/else 模拟 Socratic 追问, 不调任何 LLM API。*

## Pre-Tutorial Task (强制提取练习, 提交前必做)

> Butler (2010) 检索练习证据: 提取 (retrieval) 比重学 (restudy) 长期保留高 40%+。
> 进 tutorial 前, 你必须先提交以下三件 artifact (写在 `pre_tutorial.md`):

1. **代码**: 用 `%timeit` 在 10 万级 list 和 dict 上各跑 `"P50000" in container`, 贴出 ns 量级对比
2. **论证**: 200 字阐述"为什么 product_id 查询必须用 dict 而非 list", 必须包含 hash table / O(1) / O(n) 三个关键词
3. **反例**: 给出一个 dict 反而不如 list 的场景 (提示: 数据规模? 查询模式?)

**未提交者, tutorial 拒绝开始** (Oxford tutorial 强制预习传统)。

提交后, 本 notebook 的 Socratic loop (cell 3) 会基于你的 pre_tutorial.md 追问 4+ 轮。

In [ ]:
# Socratic Loop (静态 if/else 模拟, >=4 轮, >=5 苏格拉底问)
# 真实库锚点: list/dict/set/tuple/deque + collections + heapq

import json, os

# 模拟学生提交的 pre_tutorial.md 答案 (实际场景从文件读)
student_answers = {
    'q1_dict_vs_list': 'dict 比 list 快',  # vague, 触发追问
    'q2_collections': 'Counter',            # 不完整, 触发追问
    'q3_heapq_complexity': 'O(n log k)',    # 对, 但追问"为什么"
    'q4_arrow_polars': '不知道',            # defense 失败, 降脚手架
}

def socratic_round(round_num, ans):
    """静态 if/else 模拟牛津 tutorial fellow 的 Socratic 追问。
    每轮返回 1 个 probing question, 检测 defense 是否成立。
    连续 2 次 defense 失败 -> 降一级脚手架 (worked example)。
    """
    questions = [
        # 轮 1: ILO1 结构选择 - 追问 vague claim
        "Q1 (Socratic): 你说 'dict 比 list 快' -- 快多少? 在什么数据规模下? "
        "如果产品只有 50 个, dict 还快吗? hash 冲突时 dict 还是 O(1) 吗? 凭什么?",
        # 轮 2: ILO2 collections - 追问不完整答案
        "Q2 (Socratic): 你说 'Counter' -- Counter 继承自什么? "
        "它的 __missing__ 如何实现自动初始化? 为什么 defaultdict(list) 比 dict.setdefault 更 Pythonic? "
        "如果让你按 channel 分组 1k 订单, 你会怎么写?",
        # 轮 3: ILO3 heapq - 追问复杂度来历
        "Q3 (Socratic): 你答 'O(n log k)' -- 这个 log k 从哪来? "
        "堆的大小是多少? 每个元素进出堆的代价? "
        "为什么 k=10, n=10^6 时 heapq.nlargest 比 sorted()[:10] 快? 反例: k 接近 n 时呢?",
        # 轮 4: 前沿 Arrow - defense 失败, 降脚手架
        "Q4 (Socratic, devil's advocate): 你说 '不知道 Apache Arrow' -- "
        "那 list-of-dicts (行式) 和 'dict-of-lists' (列式) 在统计所有订单金额时, 哪个快? 为什么? "
        "(提示: 缓存局部性) 这就是 Arrow 列式内存的直觉。Polars 懒求值的 query graph 是什么数据结构?",
        # 轮 5: OSF 可复现 - 追问数据治理
        "Q5 (Socratic, Feed Forward): 你用 namedtuple 定义 Product schema -- "
        "为什么不可变? 这与 OSF 可复现研究的 'fixed random seed + immutable data' 原则如何呼应? "
        "如果下游 Day 5 DVC 要追踪数据版本, 你的 schema 设计如何支持?",
    ]
    return questions[min(round_num - 1, len(questions) - 1)]


# 模拟 5 轮 Socratic loop (>=4)
transcript = []
scaffold_level = 0  # 0=追问, 1=提示, 2=worked example
fail_streak = 0

for r in range(1, 6):
    q = socratic_round(r, student_answers)
    transcript.append(f"\n=== Round {r} (scaffold=L{scaffold_level}) ===")
    transcript.append(f"Fellow: {q}")
    # 静态模拟学生 defense 是否成立
    if r == 1 and student_answers['q1_dict_vs_list'] == 'dict 比 list 快':
        transcript.append("Student: (vague) 嗯... 大概快很多?")
        transcript.append("Fellow: (reject vague) '大概快很多' 不是论证。回去用 %timeit 实测, 给我具体数字。")
        fail_streak += 1
    elif r == 2 and student_answers['q2_collections'] == 'Counter':
        transcript.append("Student: Counter 能计数...")
        transcript.append("Fellow: (probe) Counter 继承自 dict, __missing__ 返回 0。defaultdict 呢? 它的 __missing__ 调用什么?")
        fail_streak += 1
    elif r == 3:
        transcript.append("Student: log k 因为堆大小是 k, 每元素 sift-up/down 是 O(log k)...")
        transcript.append("Fellow: (validate) 对。那 k 接近 n 时, heapq 还有优势吗? 反例在哪?")
        fail_streak = 0  # defense 部分成立
    elif r == 4:
        transcript.append("Student: 不知道 Arrow...")
        transcript.append("Fellow: (scaffold fade -> worked example) 列式 = dict-of-lists, 统计单列只读该列, 缓存局部性 10-100x。这是 Arrow 直觉。重做 reading.md Arrow 条目。")
        fail_streak += 1
        scaffold_level = min(scaffold_level + 1, 2)
    else:
        transcript.append("Student: namedtuple 不可变 = 可哈希 = 可作 dict 键, 支持 DVC 版本追踪...")
        transcript.append("Fellow: (validate + Feed Forward) 很好。下一步: Day 3 SQL 的 JOIN 本质就是 dict 查询的扩展。")
        fail_streak = 0
    if fail_streak >= 2:
        scaffold_level = min(scaffold_level + 1, 2)
        transcript.append(f"Fellow: (weak_loop trigger) 连续 {fail_streak} 次失败, 降脚手架到 L{scaffold_level}。回 practice.md drill D{max(1, r-1)} 的 Worked 阶段重看。")
        fail_streak = 0

print('\n'.join(transcript))
print('\n--- Socratic loop 结束, 共 5 轮, >=5 个 probing questions ---')

In [ ]:
# student_model.json 读写 - 跨单元复用, 记录掌握度/盲点
# 牛津 tutorial fellow 的 mental model 显式化

import json, os
from datetime import datetime

MODEL_PATH = 'student_model.json'

def load_student_model():
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, encoding='utf-8') as f:
            return json.load(f)
    # 初始化 - Day 2 数据结构
    return {
        'unit': 'skill-0-business-analytics/day-2-data-structures',
        'mastery': {
            'S1-结构选择': 0.0,   # 0.0-1.0, 0.8 = mastery
            'S2-collections聚合': 0.0,
            'S3-高级结构前沿': 0.0,
        },
        'blind_spots': [],   # 盲点列表, 用于 exit artifact
        'weak_drills': [],   # 触发过 weak_loop 的 drill_id
        'scaffold_level': 0, # 当前脚手架等级 (0=追问, 1=提示, 2=worked)
        'last_tutorial': None,  # ISO 日期, 用于限频
        'history': []
    }

def update_student_model(model, round_num, defense_ok, blind_spot=None):
    """根据本轮 defense 更新 model."""
    skill_map = {1: 'S1-结构选择', 2: 'S2-collections聚合', 3: 'S3-高级结构前沿',
                 4: 'S3-高级结构前沿', 5: 'S3-高级结构前沿'}
    skill = skill_map.get(round_num, 'S1-结构选择')
    delta = 0.15 if defense_ok else -0.10
    model['mastery'][skill] = max(0.0, min(1.0, model['mastery'][skill] + delta))
    if blind_spot:
        model['blind_spots'].append({'round': round_num, 'spot': blind_spot, 'ts': datetime.now().isoformat()})
    model['last_tutorial'] = datetime.now().date().isoformat()
    model['history'].append({'round': round_num, 'defense_ok': defense_ok, 'ts': datetime.now().isoformat()})
    return model

def save_student_model(model):
    with open(MODEL_PATH, 'w', encoding='utf-8') as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

# 模拟本次 tutorial 后更新
model = load_student_model()
model = update_student_model(model, round_num=1, defense_ok=False, blind_spot='dict vs list vague, 未用 %timeit 实测')
model = update_student_model(model, round_num=2, defense_ok=False, blind_spot='Counter __missing__ 机制不清')
model = update_student_model(model, round_num=3, defense_ok=True)
model = update_student_model(model, round_num=4, defense_ok=False, blind_spot='Apache Arrow 列式内存直觉缺失')
model = update_student_model(model, round_num=5, defense_ok=True)
save_student_model(model)

print('student_model.json 已更新:')
print(json.dumps(model, ensure_ascii=False, indent=2))

## Hattie 四级形成性反馈 (Formative Feedback)

> Hattie (2007 RER 77(1):81-112) 四级反馈, 避免无效的 Self 级表扬 (如"你真聪明")。
> 本 tutorial 每轮反馈必须落在这 4 级之一, 优先 FEED-FORWARD。

- **[TASK]** 任务级 - 关于任务本身的对错 (e.g. "你的 hash table 论证错了, dict 在 hash 冲突时退化为 O(n)")
- **[PROCESS]** 过程级 - 关于解题策略 (e.g. "你没做 %timeit 实测就下结论, 这是不科学的; 下次先用 %timeit 再论证")
- **[SELF-REG]** 自我调节级 - 关于元认知 (e.g. "你能否察觉自己答的是 vague claim? 下次听到自己说'大概'时, 立即停下去查文档")
- **[FEED-FORWARD]** 前馈级 - 关于下一步 (e.g. "基于你的盲点, 下次复习 schedule.json C1 (list vs dict), 然后进 Day 3 SQL")

**禁止 Self 级表扬**: "你真聪明""学得不错" -- Hattie 元分析显示 Self 级反馈效应量 d=0.14 (几乎无效), 而 FEED-FORWARD d=0.71 (高效)。

In [ ]:
# Hattie 四级反馈 - 基于本次 5 轮 Socratic 生成
import json

feedback = {
    'round_1': {
        '[TASK]': '答 "dict 比 list 快" 是 vague claim, 未给规模/量级; hash 冲突时 dict 退化为 O(n) 未提及。',
        '[PROCESS]': '解题策略错: 未先用 %timeit 实测就下结论。正确策略: 实测 -> 量化 -> 论证。',
        '[SELF-REG]': '你能否察觉自己说 "大概快很多" 是 vague? 下次听到自己说 "大概/应该/好像" 时, 立即停下去查文档。',
        '[FEED-FORWARD]': '回 practice.md drill D1 Worked 阶段, 重看 hash table 实现; 复习 schedule.json C1 (1 天后第 1 次复习)。',
    },
    'round_2': {
        '[TASK]': '答 "Counter" 不完整; 未提 Counter 继承 dict, 未提 defaultdict 的 __missing__ 调用 list()。',
        '[PROCESS]': '记忆碎片化: 知道名字不知道机制。策略: 看 collections 源码 (Lib/collections/__init__.py)。',
        '[SELF-REG]': '你能区分 "知道名字" 和 "知道机制" 吗? 前者是 recall, 后者是 understand (Bloom 2 vs 3)。',
        '[FEED-FORWARD]': '回 practice.md drill D2 Worked; 复习 schedule.json C2 (1 天后); 进 Day 4 pandas 向量化前必须 mastery >=70%。',
    },
    'round_3': {
        '[TASK]': '答 "O(n log k)" 正确; log k 来自堆大小 k, 每元素 sift-up/down O(log k)。',
        '[PROCESS]': '反例 (k 接近 n) 未主动给出, 被动答对。策略: 主动找反例是 mastery 的标志。',
        '[SELF-REG]': '你能在解题后主动问 "反例在哪" 吗? 这是 expert vs novice 的关键差异。',
        '[FEED-FORWARD]': 'heapq 已 mastery, 进 schedule.json C3 (3 天后复习); 准备 Day 5 DVC 数据版本管理。',
    },
    'round_4': {
        '[TASK]': 'Apache Arrow 直觉缺失; 列式 vs 行式的缓存局部性未提及。',
        '[PROCESS]': '前沿点依赖记忆而非推理。策略: 从 list-of-dicts (行式) 推导 dict-of-lists (列式) 的性能差异。',
        '[SELF-REG]': '你遇到 "不知道" 时, 能否从已知推未知? 这是 Vygotsky 最近发展区的运用。',
        '[FEED-FORWARD]': '读 reading.md Apache Arrow 条目; 复习 schedule.json C4 (1 天后); poster 阶段必须用 %timeit 实测 Arrow vs list-of-dicts。',
    },
    'round_5': {
        '[TASK]': 'namedtuple 不可变 = 可哈希 = 可作 dict 键, 支持 DVC 版本追踪 -- 论证成立。',
        '[PROCESS]': '从工程权衡推导数据治理, 跨域连接好。',
        '[SELF-REG]': '保持这种 "工程 -> 治理" 的跨域推理习惯。',
        '[FEED-FORWARD]': '进 Day 3 SQL, JOIN 本质是 dict 查询扩展; Day 5 DVC 数据血缘追踪; OSF 注册 schema。',
    },
}
print(json.dumps(feedback, ensure_ascii=False, indent=2))

## 限频与 Exit Artifact (防依赖 + 闭环)

### 限频 (Usage Limit)

- **本单元 tutorial 限频: 1 次/天** -- 防止学生把 LLM 当答案机, 强制自行建构 (Vygotsky 共构 vs 获取)。
- 实现见 cell 4 `student_model.json` 的 `last_tutorial` 字段; 同一天再次启动 tutorial, fellow 返回: "你今天已经 tutorial 过了, 先去练 practice.md drill D{n}, 明天再来。"
- **绕过限频的检测**: 若 history 中同一天 >=2 次, 自动降 scaffold_level 至 2 (worked example only), 不再 Socratic 追问。

### Exit Artifact (Tutorial 闭环交付物)

每次 tutorial 结束, 必须提交以下 3 项, 写入 `exit_artifact.md`:

1. **2-3 个盲点** (从 student_model.json blind_spots 提取):
   - 本 Day 盲点示例: 
     - "dict vs list 性能论证未用 %timeit 实测"
     - "Counter __missing__ 机制不清"
     - "Apache Arrow 列式内存直觉缺失"

2. **推荐复习单元** (基于盲点跨单元映射):
   - 盲点 1 -> 复习 Day 1 Python 基础 (list/dict 语法) + 本 Day practice.md D1
   - 盲点 2 -> 复习本 Day practice.md D2 + reading.md collections 条目
   - 盲点 3 -> 复习本 Day reading.md Apache Arrow + 进 Day 5 数据管理 (DVC/Arrow)

3. **下次 tutorial 的 1 个 focus question** (学生自拟, 必须是基于本次盲点的延伸):
   - 示例: "如果产品只有 50 个, dict vs list 的转折点在哪?" / "Polars lazy 比 pandas eager 快多少? 在什么 query 复杂度下?"

### 闭环校验 (mastery gate)

- Exit artifact 提交后, schedule.json 中 C1-C4 的 next due 自动设为 [1, 3, 8, 21, 60, 180] 起算
- 进 Day 3 SQL 前提: ILO1 >=80% AND ILO2 >=70% AND ILO3 能独立解 (见 alignment.md mastery_threshold)
- 不达标: 回 practice.md weak_loop, schedule.json 复习间隔减半 (FSRS request_retention 临时降至 0.85)

---

*本 notebook 是 v6.0 学习科学层的牛津 tutorial 仿真, 与 v5.0 starter/solution.ipynb 互补: starter 练手, tutorial 练脑 (口头辩护 + 元认知)。*